# 03 · Bronze Delta tables
Turn raw JSONL files in the Volume into **Delta tables** (Parquet + transaction log).
Each row gets lineage columns: `_source_file`, `_run_date`, `_ingested_at`.
Bronze = raw content, **not cleaned**. Cleaning happens in Silver.

In [0]:
-- =====================================================================
-- CELL 1: Raw JSONL files -> BRONZE Delta tables
-- =====================================================================
-- read_files() = reads every file in a folder (including run_date=... subfolders)
--   and INFERS the schema (column names + types) from the JSON.
-- _metadata.file_path = a hidden column Databricks gives every file read:
--   the full path of the file each row came from (LINEAGE).
-- regexp_extract(text, pattern, 1) = pull a piece out of text.
--   Here: the date after "run_date=" in the file path.
-- CREATE OR REPLACE = idempotent. Each reload becomes a new table VERSION,
--   so we can still time travel to older loads.
-- _rescued_data (added automatically by read_files) = a safety net column:
--   any value that didn't fit the inferred schema lands there instead of being lost.

CREATE OR REPLACE TABLE procintel.bronze.projects
COMMENT 'RAW World Bank projects for Kenya, as returned by search.worldbank.org/api/v2/projects'
AS SELECT *,
       _metadata.file_path                                        AS _source_file,
       regexp_extract(_metadata.file_path, 'run_date=([0-9-]+)', 1) AS _run_date,
       current_timestamp()                                        AS _ingested_at
FROM read_files('/Volumes/procintel/bronze/raw_files/projects/', format => 'json');

CREATE OR REPLACE TABLE procintel.bronze.contracts
COMMENT 'RAW contract awards for Kenya, from search.worldbank.org/api/contractdata (source: STEP)'
AS SELECT *,
       _metadata.file_path                                        AS _source_file,
       regexp_extract(_metadata.file_path, 'run_date=([0-9-]+)', 1) AS _run_date,
       current_timestamp()                                        AS _ingested_at
FROM read_files('/Volumes/procintel/bronze/raw_files/contracts/', format => 'json');

CREATE OR REPLACE TABLE procintel.bronze.notices
COMMENT 'RAW procurement notices for Kenya projects, from search.worldbank.org/api/v2/procnotices'
AS SELECT *,
       _metadata.file_path                                        AS _source_file,
       regexp_extract(_metadata.file_path, 'run_date=([0-9-]+)', 1) AS _run_date,
       current_timestamp()                                        AS _ingested_at
FROM read_files('/Volumes/procintel/bronze/raw_files/notices/', format => 'json');

CREATE OR REPLACE TABLE procintel.bronze.documents
COMMENT 'RAW document METADATA for Kenya (ICR, PAD, ISR...), from search.worldbank.org/api/v2/wds. Full text loaded later.'
AS SELECT *,
       _metadata.file_path                                        AS _source_file,
       regexp_extract(_metadata.file_path, 'run_date=([0-9-]+)', 1) AS _run_date,
       current_timestamp()                                        AS _ingested_at
FROM read_files('/Volumes/procintel/bronze/raw_files/documents/', format => 'json');

CREATE OR REPLACE TABLE procintel.bronze.indicators
COMMENT 'RAW governance indicators for Kenya: CPIA (1-6) and WGI 2025 revision (-2.5..+2.5)'
AS SELECT *,
       _metadata.file_path                                        AS _source_file,
       regexp_extract(_metadata.file_path, 'run_date=([0-9-]+)', 1) AS _run_date,
       current_timestamp()                                        AS _ingested_at
FROM read_files('/Volumes/procintel/bronze/raw_files/indicators/', format => 'json');

In [0]:
-- =====================================================================
-- CELL 2: RECONCILIATION -> table row counts must equal the files we landed
-- =====================================================================
-- UNION ALL stacks the results of several SELECTs into one table.
-- Expected: projects 282, contracts 6661, notices 7251, documents 8758, indicators 147

SELECT 'projects'   AS table_name, COUNT(*) AS row_count FROM procintel.bronze.projects
UNION ALL SELECT 'contracts',  COUNT(*) FROM procintel.bronze.contracts
UNION ALL SELECT 'notices',    COUNT(*) FROM procintel.bronze.notices
UNION ALL SELECT 'documents',  COUNT(*) FROM procintel.bronze.documents
UNION ALL SELECT 'indicators', COUNT(*) FROM procintel.bronze.indicators;

In [0]:
-- =====================================================================
-- CELL 3: DELTA LAKE -> the table's change history (from the _delta_log)
-- =====================================================================
-- Every write = one VERSION. Shows who did it, when, what operation, and metrics.
-- Run Cell 1 again later and you'll see version 1, 2, ... appear here.
-- Time travel example:  SELECT * FROM procintel.bronze.contracts VERSION AS OF 0

DESCRIBE HISTORY procintel.bronze.contracts;

In [0]:
-- =====================================================================
-- CELL 4: INSPECT contracts -> what types did read_files infer?
-- =====================================================================
-- typeof(x) returns the data type of a value: string, bigint, array<string>...
-- We look at dates and amounts, because those need converting in Silver.

SELECT contr_id, projectid, contr_sgn_date, contr_no_obj_dat,
       total_contr_amnt, supplier_contr_amount, supp_name,
       procu_meth_text, procurement_group, rvw_type,
       typeof(contr_sgn_date)        AS type_sign_date,
       typeof(total_contr_amnt)      AS type_amount,
       typeof(supplier_contr_amount) AS type_supplier_amount,
       typeof(supp_name)             AS type_supplier
FROM procintel.bronze.contracts
LIMIT 5;

In [0]:
-- =====================================================================
-- CELL 5: INSPECT notices -> dates and notice types
-- =====================================================================
SELECT id, project_id, notice_type, noticedate, submission_deadline_date,
       procurement_method_name, procurement_group, bid_reference_no,
       typeof(noticedate) AS type_noticedate
FROM procintel.bronze.notices
LIMIT 5;

In [0]:
-- =====================================================================
-- CELL 6: INSPECT documents -> which document TYPES do we have for Kenya?
-- =====================================================================
-- We only need a few types for the chatbot's "lessons learned":
--   ICR (completion reports), PAD (appraisal), ISR (status reports),
--   PPAR (independent evaluation), procurement plans.
-- GROUP BY counts rows per document type.

SELECT docty, COUNT(*) AS n_docs
FROM procintel.bronze.documents
GROUP BY docty
ORDER BY n_docs DESC
LIMIT 25;

In [0]:
-- =====================================================================
-- CELL 7: Which document columns hold the PROJECT ID and the TEXT/PDF URLs?
-- =====================================================================
-- information_schema = Unity Catalog's built-in "table about tables".
-- Here we ask it for column names, instead of scrolling through 50+ columns.

SELECT column_name, data_type
FROM procintel.information_schema.columns
WHERE table_schema = 'bronze'
  AND table_name   = 'documents'
  AND (column_name ILIKE '%url%' OR column_name ILIKE '%proj%' OR column_name ILIKE '%date%' OR column_name ILIKE '%dt%')
ORDER BY column_name;